In [78]:
from notebook_framework import (
    NotebookProject,
)

df1=NotebookProject().load_data("notebook_env_demo.csv")

print(df1)



           date  source region  channel  visits  orders  revenue  conversion
0    2026-01-01  search  north  organic     836      71  3474.88      0.0849
1    2026-01-01     ads  north     paid     924      60  2233.92      0.0649
2    2026-01-01  social  north   social     731      32   890.48      0.0438
3    2026-01-01   email  north    email     676      74  4346.08      0.1095
4    2026-01-02  search  south  organic     786      66  3230.88      0.0840
..          ...     ...    ...      ...     ...     ...      ...         ...
115  2026-01-29   email  south    email    1183     130  7634.64      0.1099
116  2026-01-30  search   east  organic    1504     127  6216.32      0.0844
117  2026-01-30     ads   east     paid    1732     112  4170.56      0.0647
118  2026-01-30  social   east   social    1409      63  1750.72      0.0447
119  2026-01-30   email   east    email    1359     149  8750.72      0.1096

[120 rows x 8 columns]


In [79]:

df1.groupby("source").agg(visits_mean=("visits","mean"),visits_sum=("visits","sum"))


,visits_mean,visits_sum
source,,
ads,1297.800000,38934
email,995.966667,29879
search,1142.433333,34273
social,1046.533333,31396


In [80]:
df1.duplicated()

0      False
1      False
2      False
3      False
4      False
       ...  
115    False
116    False
117    False
118    False
119    False
Length: 120, dtype: bool

In [81]:
from pathlib import Path
import importlib.util
import pandas as pd

# 1) 加载 notebook_framework.py
module_path = Path(r"d:\AI-WORKSPACE\projects\crawler\data-analysis\notebooks\notebook_framework.py")
spec = importlib.util.spec_from_file_location("notebook_framework", module_path)
mod = importlib.util.module_from_spec(spec)
spec.loader.exec_module(mod)

project = mod.NotebookProject()

# 2) query：把 DataFrame 注册为 DuckDB 表，再执行 SQL
sales = pd.DataFrame(
    {
        "region": ["East", "West", "East"],
        "amount": [100, 250, 80],
    }
)

df1 = project.query(
    """
    SELECT region, SUM(amount) AS total_amount
    FROM sales
    GROUP BY region
    ORDER BY total_amount DESC
    """,
    tables={"sales": sales},
)
print("query:")
print(df1)

# 3) query_df：直接把一个 DataFrame 注册成表名 data
df2 = project.query_df(
    """
    SELECT *
    FROM data
    WHERE amount >= 100
    """,
    sales,
name="data",
)
print("\nquery_df:")
print(df2)

# 4) query_file：直接从文件读取并执行 SQL
df3 = project.query_file(
    """
    SELECT * FROM read_csv_auto(?)
    LIMIT 5
    """,
    "raw/news.csv",
)
print("\nquery_file:")
print(df3.head())

# 5) query_csv：先创建 VIEW，再查询
df4 = project.query_csv(
    """
    SELECT * FROM data
    LIMIT 5
    """,
    "raw/news.csv",
)
print("\nquery_csv:")
print(df4.head())

query:
  region  total_amount
0   West         250.0
1   East         180.0

query_df:
  region  amount
0   East     100
1   West     250

query_file:
                           name   price
0            A Light in the ...  £51.77
1            Tipping the Velvet     NaN
2                    Soumission  £50.10
3                 Sharp Objects  £47.82
4  Sapiens: A Brief History ...  £54.23


BinderException: Binder Error: Unexpected prepared parameter. This type of statement can't be prepared!